# berts-gg-qa-training

Offline **training-only** notebook for Google QUEST Q&A Labeling.

- Internet-off compatible (Kaggle datasets only).
- Ensemble of RoBERTa-base, RoBERTa-large, DeBERTa-v3-base.
- Regression for all targets.
- Twin dual-input architecture with weighted multi-head fusion.
- Postprocess clipping + snapping to known labels.
- Output exactly 3 SWA checkpoints (no fold).

In [ ]:
import os, gc, json, html, re, random
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

In [ ]:
@dataclass
class CFG:
    seed: int = 42
    epochs: int = 3
    train_bs: int = 4
    valid_bs: int = 8
    grad_accum: int = 2
    lr: float = 2e-5
    wd: float = 0.01
    max_len_q: int = 256
    max_len_a: int = 256
    num_workers: int = 2
    use_amp: bool = True
    use_swa: bool = True
    swa_start_epoch: int = 1
    val_group_frac: float = 0.10
    out_dir: str = '/kaggle/working/ggqa_bert_models'
    model_zoo = {
        'roberta_base': '/kaggle/input/datasets/abhishek/roberta-base',
        'roberta_large': '/kaggle/input/datasets/marshal02/robertalarge',
        'deberta_v3_base': '/kaggle/input/datasets/debarshichanda/debertav3base'
    }

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
os.makedirs(f'{cfg.out_dir}/checkpoints', exist_ok=True)
os.makedirs(f'{cfg.out_dir}/artifacts', exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
puncts = [',', '.', '"', ':', ')', '(', '-', '!', '?', '|', ';', "'", '$', '&', '/', '[', ']', '>', '%', '=', '#', '*', '+', '\\', '•', '~', '@', '£', '·', '_', '{', '}', '©', '^', '®', '`', '<', '→', '°', '€', '™', '♥', '←', '×', '§', '…', '\n', '\xa0', '\t', '“', '”', '–', '—']
mispell_dict = {"can't": "cannot", "dont": "do not", "don't": "do not", "doesnt": "does not", "doesn't": "does not", "isn't": "is not", "aren't": "are not", "it's": "it is", "i'm": "i am", "you're": "you are", "they're": "they are"}

def clean_text(x):
    x = str(x)
    for p in puncts: x = x.replace(p, f' {p} ')
    return x

def clean_numbers(x):
    x = re.sub('[0-9]{5,}', '#####', x); x = re.sub('[0-9]{4}', '####', x); x = re.sub('[0-9]{3}', '###', x); x = re.sub('[0-9]{2}', '##', x)
    return x

def _get_mispell(d):
    return d, re.compile('(%s)' % '|'.join(map(re.escape, d.keys())))

def replace_typical_misspell(text):
    d, rp = _get_mispell(mispell_dict)
    return rp.sub(lambda m: d[m.group(0)], text)

def clean_data(df, cols):
    for c in cols:
        df[c] = df[c].astype(str).map(html.unescape).apply(clean_numbers).str.lower().apply(clean_text).apply(replace_typical_misspell)
    return df

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/google-quest-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/google-quest-challenge/test.csv')
text_cols = ['question_title','question_body','answer']
train[text_cols] = train[text_cols].fillna('')
test[text_cols] = test[text_cols].fillna('')
train = clean_data(train, text_cols)
test = clean_data(test, text_cols)

target_cols = [c for c in train.columns if c not in test.columns and c != 'qa_id']
question_targets = [c for c in target_cols if c.startswith('question_')]
answer_targets = [c for c in target_cols if c.startswith('answer_')]
q_idx = [target_cols.index(c) for c in question_targets]
a_idx = [target_cols.index(c) for c in answer_targets]
y = train[target_cols].values.astype(np.float32)
label_values = {c: np.sort(train[c].unique()) for c in target_cols}
groups = train['question_body'].values
print('targets', len(target_cols), 'q', len(question_targets), 'a', len(answer_targets))

In [ ]:
def mean_spearman(y_true, y_pred):
    ss=[]
    for i in range(y_true.shape[1]):
        s=spearmanr(y_true[:,i], y_pred[:,i]).correlation
        ss.append(0.0 if np.isnan(s) else s)
    return float(np.mean(ss))

CLIPPINGS = {
 'question_has_commonly_accepted_answer': (0.0,0.6),
 'question_conversational': (0.15,1.0),
 'question_multi_intent': (0.1,1.0),
 'question_type_choice': (0.1,1.0),
 'question_type_compare': (0.1,1.0),
 'question_type_consequence': (0.08,1.0),
 'question_type_definition': (0.1,1.0),
 'question_type_entity': (0.13,1.0),
}

def apply_clipping(preds, cols):
    p=preds.copy()
    for j,c in enumerate(cols):
        if c in CLIPPINGS: p[:,j]=np.clip(p[:,j], CLIPPINGS[c][0], CLIPPINGS[c][1])
    return p

def snap_to_known_labels(preds, cols, lv):
    out=np.zeros_like(preds)
    for j,c in enumerate(cols):
        vals=lv[c]
        idx=np.abs(preds[:,[j]]-vals.reshape(1,-1)).argmin(axis=1)
        out[:,j]=vals[idx]
    return out

def ensure_non_constant_columns(preds):
    out=preds.copy(); n=len(out)
    if n<=1: return out
    tiny=np.linspace(0.0,1e-7,n,dtype=np.float32)
    for j in range(out.shape[1]):
        if np.allclose(out[:,j], out[0,j]): out[:,j]=out[:,j]+tiny
    return out

In [ ]:
class QADataset(Dataset):
    def __init__(self, df, y, tok, max_len_q=256, max_len_a=256):
        self.df=df; self.y=y; self.tok=tok; self.max_len_q=max_len_q; self.max_len_a=max_len_a
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r=self.df.iloc[i]
        qtxt=f"{r['question_title']} [SEP] {r['question_body']}"
        atxt=f"{r['question_title']} [SEP] {r['answer']}"
        q=self.tok(qtxt, truncation=True, max_length=self.max_len_q, padding='max_length', return_tensors='pt')
        a=self.tok(atxt, truncation=True, max_length=self.max_len_a, padding='max_length', return_tensors='pt')
        item={'q_input_ids':q['input_ids'].squeeze(0),'q_attention_mask':q['attention_mask'].squeeze(0),'a_input_ids':a['input_ids'].squeeze(0),'a_attention_mask':a['attention_mask'].squeeze(0)}
        if self.y is not None: item['labels']=torch.tensor(self.y[i], dtype=torch.float)
        return item

In [ ]:
class TwinEnsembleRegressor(nn.Module):
    def __init__(self, model_path, n_targets, q_out, a_out):
        super().__init__()
        self.q_encoder=AutoModel.from_pretrained(model_path, local_files_only=True)
        self.a_encoder=AutoModel.from_pretrained(model_path, local_files_only=True)
        h=self.q_encoder.config.hidden_size
        self.head_a=nn.Sequential(nn.Linear(h*2,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,n_targets))
        self.head_q=nn.Sequential(nn.Linear(h,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,q_out))
        self.head_a_only=nn.Sequential(nn.Linear(h,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,a_out))
        self.fusion=nn.Sequential(nn.Linear(h*4,h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h,n_targets))
    def pooled(self, encoder, ids, mask):
        out=encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        m=mask.unsqueeze(-1)
        return (out*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self, q_ids, q_mask, a_ids, a_mask):
        qv=self.pooled(self.q_encoder,q_ids,q_mask); av=self.pooled(self.a_encoder,a_ids,a_mask)
        pred_a=self.head_a(torch.cat([qv,av],dim=1))
        pred_b=torch.cat([self.head_q(qv), self.head_a_only(av)], dim=1)
        pred_c=self.fusion(torch.cat([qv,av,torch.abs(qv-av),qv*av], dim=1))
        return 0.25*pred_a + 0.25*pred_b + 0.5*pred_c

In [ ]:
def run_single_split(model_name, model_path, tr_idx, va_idx):
    tok=AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    tr_ds=QADataset(train.iloc[tr_idx].reset_index(drop=True), y[tr_idx], tok, cfg.max_len_q, cfg.max_len_a)
    va_ds=QADataset(train.iloc[va_idx].reset_index(drop=True), y[va_idx], tok, cfg.max_len_q, cfg.max_len_a)
    te_ds=QADataset(test.reset_index(drop=True), None, tok, cfg.max_len_q, cfg.max_len_a)
    tr_dl=DataLoader(tr_ds,batch_size=cfg.train_bs,shuffle=True,num_workers=cfg.num_workers,pin_memory=True)
    va_dl=DataLoader(va_ds,batch_size=cfg.valid_bs,shuffle=False,num_workers=cfg.num_workers,pin_memory=True)
    te_dl=DataLoader(te_ds,batch_size=cfg.valid_bs,shuffle=False,num_workers=cfg.num_workers,pin_memory=True)

    model=TwinEnsembleRegressor(model_path, len(target_cols), len(q_idx), len(a_idx)).to(device)
    opt=torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    total_steps=max(1,len(tr_dl)*cfg.epochs//cfg.grad_accum)
    sch=get_linear_schedule_with_warmup(opt, int(0.1*total_steps), total_steps)
    scaler=GradScaler(enabled=cfg.use_amp)
    crit=nn.SmoothL1Loss()

    swa_state=None; swa_count=0
    best=-1e9
    best_path=f"{cfg.out_dir}/checkpoints/{model_name}_best.pt"
    swa_path=f"{cfg.out_dir}/checkpoints/{model_name}_swa.pt"

    for ep in range(cfg.epochs):
        model.train(); opt.zero_grad(set_to_none=True)
        for step,b in enumerate(tr_dl):
            q_ids=b['q_input_ids'].to(device); q_m=b['q_attention_mask'].to(device)
            a_ids=b['a_input_ids'].to(device); a_m=b['a_attention_mask'].to(device)
            lbl=b['labels'].to(device)
            with autocast(enabled=cfg.use_amp):
                pred=model(q_ids,q_m,a_ids,a_m)
                loss=crit(pred,lbl)/cfg.grad_accum
            scaler.scale(loss).backward()
            if (step+1)%cfg.grad_accum==0:
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()

        if cfg.use_swa and ep>=cfg.swa_start_epoch:
            cur={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            if swa_state is None: swa_state=cur
            else:
                for k in swa_state: swa_state[k]=(swa_state[k]*swa_count+cur[k])/(swa_count+1)
            swa_count += 1

        model.eval(); va_pred=[]
        with torch.no_grad():
            for b in va_dl:
                q_ids=b['q_input_ids'].to(device); q_m=b['q_attention_mask'].to(device)
                a_ids=b['a_input_ids'].to(device); a_m=b['a_attention_mask'].to(device)
                va_pred.append(model(q_ids,q_m,a_ids,a_m).detach().cpu().numpy())
        va_pred=np.concatenate(va_pred)
        score=mean_spearman(y[va_idx], va_pred)
        print(model_name, 'epoch', ep, 'cv', round(score,6))
        if score>best: best=score; torch.save(model.state_dict(), best_path)

    if cfg.use_swa and swa_state is not None: torch.save(swa_state, swa_path)
    else: torch.save(torch.load(best_path, map_location='cpu'), swa_path)

    model.load_state_dict(torch.load(swa_path, map_location=device)); model.eval()
    oof=[]; test_pred=[]
    with torch.no_grad():
        for b in va_dl:
            q_ids=b['q_input_ids'].to(device); q_m=b['q_attention_mask'].to(device)
            a_ids=b['a_input_ids'].to(device); a_m=b['a_attention_mask'].to(device)
            oof.append(model(q_ids,q_m,a_ids,a_m).detach().cpu().numpy())
        for b in te_dl:
            q_ids=b['q_input_ids'].to(device); q_m=b['q_attention_mask'].to(device)
            a_ids=b['a_input_ids'].to(device); a_m=b['a_attention_mask'].to(device)
            test_pred.append(model(q_ids,q_m,a_ids,a_m).detach().cpu().numpy())
    oof=np.concatenate(oof); test_pred=np.concatenate(test_pred)
    del model, tr_dl, va_dl, te_dl; gc.collect(); torch.cuda.empty_cache()
    return oof, test_pred, best, swa_path

In [ ]:
unique_groups=pd.Series(groups).drop_duplicates().values
rng=np.random.default_rng(cfg.seed)
shuffled=rng.permutation(unique_groups)
val_size=max(1,int(cfg.val_group_frac*len(shuffled)))
val_groups=set(shuffled[:val_size])
va_idx=np.where(pd.Series(groups).isin(val_groups).values)[0]
tr_idx=np.where(~pd.Series(groups).isin(val_groups).values)[0]

all_model_oof={}; all_model_test={}; model_paths={}
for model_name, model_path in cfg.model_zoo.items():
    print('\n===', model_name, '===')
    oof=np.zeros((len(train), len(target_cols)), dtype=np.float32)
    fold_oof, fold_test, fold_score, ckpt = run_single_split(model_name, model_path, tr_idx, va_idx)
    oof[va_idx]=fold_oof
    all_model_oof[model_name]=oof
    all_model_test[model_name]=fold_test
    model_paths[model_name]=ckpt
    print('saved swa checkpoint:', ckpt)

ensemble_oof=np.mean(np.stack(list(all_model_oof.values()), axis=0), axis=0)
ensemble_test=np.mean(np.stack(list(all_model_test.values()), axis=0), axis=0)
print('Ensemble raw CV:', mean_spearman(y[va_idx], ensemble_oof[va_idx]))

In [ ]:
pp_clip=apply_clipping(ensemble_oof, target_cols)
pp_snap=snap_to_known_labels(pp_clip, target_cols, label_values)
pp_snap=ensure_non_constant_columns(pp_snap)
print('CV clip:', mean_spearman(y[va_idx], pp_clip[va_idx]))
print('CV clip+snap:', mean_spearman(y[va_idx], pp_snap[va_idx]))

final_test=snap_to_known_labels(apply_clipping(ensemble_test, target_cols), target_cols, label_values)
final_test=ensure_non_constant_columns(final_test)

In [ ]:
np.save(f'{cfg.out_dir}/artifacts/oof_ensemble.npy', ensemble_oof)
np.save(f'{cfg.out_dir}/artifacts/test_ensemble.npy', ensemble_test)
np.save(f'{cfg.out_dir}/artifacts/test_final_pp.npy', final_test)
with open(f'{cfg.out_dir}/artifacts/target_cols.json','w') as f: json.dump(target_cols,f)
with open(f'{cfg.out_dir}/artifacts/label_values.json','w') as f: json.dump({k:[float(x) for x in v] for k,v in label_values.items()},f)
with open(f'{cfg.out_dir}/artifacts/clippings.json','w') as f: json.dump({k:[float(v[0]),float(v[1])] for k,v in CLIPPINGS.items()},f)
with open(f'{cfg.out_dir}/artifacts/model_paths.json','w') as f: json.dump(model_paths,f,indent=2)
print('Saved SWA checkpoints:', json.dumps(model_paths, indent=2))
print('Artifacts saved to', f'{cfg.out_dir}/artifacts')